# 📘 Módulo 14 — Transformers, Attention e o Deep Learning Moderno

Este módulo apresenta a arquitetura que mudou tudo:

**Transformers**  
(Vaswani et al., 2017 — *Attention is All You Need*)

Você aprenderá:
- por que RNNs e LSTMs não escalam
- como funciona o mecanismo de atenção
- como funciona o self‑attention
- como funciona o multi‑head attention
- como funciona o encoder e o decoder
- como funciona o positional encoding
- como treinar um Transformer simples
- como usar modelos pré‑treinados (BERT, GPT, T5)

---

# 🎯 Objetivos do Módulo 14

Ao final deste módulo, você será capaz de:

✔️ entender a motivação dos Transformers  
✔️ explicar o mecanismo de atenção  
✔️ construir um Transformer Encoder simples  
✔️ usar embeddings + positional encoding  
✔️ treinar um modelo Transformer para classificação  
✔️ entender a base dos modelos modernos de linguagem  

---

# 📚 Índice

1. [Por que Transformers? Limitações das RNNs](#motivacao)
2. [Atenção: a ideia que mudou tudo](#atencao)
3. [Self‑Attention e Multi‑Head Attention](#selfattention)
4. [Positional Encoding](#positional)
5. [Construindo um Transformer Encoder do zero](#encoder)
6. [Treinando um Transformer para classificação de texto](#treino)
7. [Transformers pré‑treinados (BERT, GPT, T5)](#pretreinados)
8. [Projeto Final — Mini‑GPT para completar frases](#projeto)

---

<a id="setup"></a>
# 0. Setup — bibliotecas

Vamos usar TensorFlow/Keras para construir Transformers.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.__version__

<a id="motivacao"></a>
# 2. Por que Transformers? As limitações das RNNs e LSTMs

Antes de 2017, praticamente todo NLP usava:
- RNN
- LSTM
- GRU
- Seq2Seq com atenção

Esses modelos funcionavam, mas tinham **limitações sérias**:

---

# 🔥 2.1 Problema 1 — Processamento sequencial (não paralelizável)

RNNs processam **um token por vez**:

```
x1 → x2 → x3 → x4 → ...
```

Isso significa:
- treinamento lento
- inferência lenta
- não aproveita GPUs de forma eficiente

Transformers resolvem isso com **atenção paralela**.

---

# 🔥 2.2 Problema 2 — Dependências longas são difíceis

Mesmo LSTMs sofrem para lembrar informações distantes.

Exemplos:
- “O filme que eu vi ontem, apesar de longo, foi **excelente**.”
- “Se o preço continuar subindo, no próximo mês teremos **alta**.”

RNNs/LSTMs esquecem o começo da frase.

Transformers resolvem isso com **self‑attention**, que conecta **todos os tokens entre si**.

---

# 🔥 2.3 Problema 3 — Gradiente que desaparece

Mesmo com LSTM/GRU, ainda existe:
- perda de memória
- dificuldade em capturar relações distantes

Transformers não têm recorrência → **não têm gradiente que desaparece**.

---

# 🔥 2.4 Problema 4 — Custo quadrático em profundidade

Para uma frase longa, RNNs precisam de:
- muitos passos
- muita profundidade

Transformers usam atenção:
- custo quadrático em **largura** (tamanho da sequência)
- mas **paralelizável**

Resultado: muito mais rápido.

---

# 🔥 2.5 Problema 5 — Dificuldade em capturar contexto global

RNNs veem o texto como:

```
passado → presente → futuro
```

Transformers veem como:

```
todos os tokens se relacionam com todos
```

Isso permite:
- entender ironia
- entender contexto global
- entender relações complexas

---

# 🔥 2.6 Visualização: RNN vs Transformer

import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))
plt.text(0.1, 0.5, "RNN → fluxo linear", fontsize=14)
plt.text(5.1, 0.5, "Transformer → conexões globais", fontsize=14)
plt.axis("off")
plt.title("Diferença conceitual entre RNN e Transformer")
plt.show()

---
# 🔥 2.7 A frase que mudou a história

Em 2017, o paper **Attention is All You Need** apresentou a ideia:

> “Não precisamos mais de recorrência.”

Isso permitiu:
- paralelização total
- modelos gigantes
- treinamento em escala
- surgimento de BERT, GPT, T5, LLaMA, etc.

Transformers são a base da IA moderna.

---

# 2.8 Conclusão desta parte

✔️ RNNs são lentas e não paralelizáveis  
✔️ LSTMs têm dificuldade com dependências longas  
✔️ Transformers eliminam recorrência  
✔️ Self‑attention conecta todos os tokens  
✔️ Isso permitiu modelos gigantes e poderosos  

Agora estamos prontos para:

**PARTE 3 — Self‑Attention e Multi‑Head Attention (o coração dos Transformers).**

<a id="selfattention"></a>
# 3. Self‑Attention e Multi‑Head Attention

Esta é a parte mais importante de todo o módulo.

Self‑Attention é o mecanismo que permite que:
- cada token "olhe" para todos os outros tokens
- o modelo entenda contexto global
- o modelo capture relações longas
- o modelo seja totalmente paralelizável

É o núcleo dos Transformers.

---
# 3.1 A ideia central do Self‑Attention

Para cada token, queremos responder:

> “Quais outros tokens são relevantes para entender este token?”

Exemplo:

> “O gato que estava no muro **caiu**.”

Para entender **caiu**, o modelo precisa olhar para **gato**, não para **muro**.

Self‑Attention faz exatamente isso.

---
# 3.2 Queries, Keys e Values (Q, K, V)

Cada token é transformado em três vetores:

- **Query (Q)** → o que este token está procurando  
- **Key (K)** → o que este token oferece  
- **Value (V)** → a informação que será combinada  

Fórmula da atenção:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V
$$

Isso gera pesos que dizem:

> “Quanto cada token deve prestar atenção aos outros?”

---
# 3.3 Visualização do Self‑Attention

import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.imshow(np.random.rand(6,6), cmap="viridis")
plt.title("Matriz de Atenção (exemplo)")
plt.colorbar()
plt.show()

🟣 **Interpretação**

Cada linha = um token  
Cada coluna = outro token  
Cores = força da atenção  

O modelo aprende padrões como:
- sujeito ↔ verbo  
- pronomes ↔ referentes  
- palavras relacionadas  
- dependências longas  

---
# 3.4 Multi‑Head Attention

Uma única atenção captura apenas um tipo de relação.

Multi‑Head Attention cria **várias atenções paralelas**, cada uma aprendendo algo diferente:

- sintaxe  
- semântica  
- dependências longas  
- relações locais  

Depois concatena tudo.

In [ ]:
plt.figure(figsize=(10,3))
plt.text(0.1, 0.5, "Head 1 → sintaxe", fontsize=14)
plt.text(3.1, 0.5, "Head 2 → semântica", fontsize=14)
plt.text(6.1, 0.5, "Head 3 → dependências longas", fontsize=14)
plt.axis("off")
plt.title("Multi‑Head Attention")
plt.show()

---
# 3.5 Implementando Self‑Attention em Keras

Vamos criar uma camada de atenção simples para entender a mecânica.

In [ ]:
class SelfAttention(layers.Layer):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.query = layers.Dense(embed_dim)
        self.key = layers.Dense(embed_dim)
        self.value = layers.Dense(embed_dim)

    def call(self, inputs):
        Q = self.query(inputs)
        K = self.key(inputs)
        V = self.value(inputs)

        scores = tf.matmul(Q, K, transpose_b=True) / tf.sqrt(tf.cast(self.embed_dim, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)
        output = tf.matmul(weights, V)
        return output

---
# 3.6 Testando a camada de Self‑Attention

In [ ]:
dummy = tf.random.normal((1, 5, 16))  # batch=1, tokens=5, embedding=16
att = SelfAttention(16)
out = att(dummy)
out.shape

🟣 **Interpretação**

A saída tem o mesmo formato da entrada:

```
(batch, tokens, embedding)
```

Mas agora cada token contém informações de todos os outros tokens.

---
# 3.7 Por que isso é revolucionário?

✔️ Paralelização total  
✔️ Captura dependências longas  
✔️ Escala para bilhões de parâmetros  
✔️ Base de BERT, GPT, T5, LLaMA, PaLM  

Self‑Attention é o mecanismo que permitiu o salto da IA moderna.

---
# 3.8 Conclusão desta parte

✔️ Entendemos Q, K, V  
✔️ Entendemos Self‑Attention  
✔️ Entendemos Multi‑Head Attention  
✔️ Implementamos atenção em Keras  
✔️ Vimos por que Transformers substituíram RNNs  

Agora estamos prontos para:

**PARTE 4 — Positional Encoding (como Transformers entendem ordem).**

<a id="positional"></a>
# 4. Positional Encoding — como Transformers entendem ordem

RNNs entendem ordem naturalmente:
- processam token por token
- mantêm estado interno

Transformers NÃO têm recorrência.

Eles processam todos os tokens **em paralelo**.

Portanto, precisam de um mecanismo explícito para representar:
- posição
- ordem
- distância entre tokens

Esse mecanismo é o **Positional Encoding**.

---
# 4.1 Por que não usar índices simples?

Poderíamos fazer:

```
posição 0 → vetor [0]
posição 1 → vetor [1]
posição 2 → vetor [2]
```

Mas isso não funciona porque:
- não captura periodicidade
- não generaliza para sequências maiores
- não cria relações suaves entre posições

Transformers precisam de algo mais rico.

---
# 4.2 A solução: funções seno e cosseno

O paper original usa:

$$
PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)
$$

$$
PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)
$$

Isso cria padrões:
- contínuos  
- suaves  
- periódicos  
- que permitem ao modelo inferir distâncias  

---
# 4.3 Visualizando Positional Encoding

import numpy as np
import matplotlib.pyplot as plt

def positional_encoding(max_len=50, d_model=16):
    pos = np.arange(max_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angles = pos / np.power(10000, (2 * (i//2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return angles

pe = positional_encoding(50, 16)

plt.figure(figsize=(10,4))
plt.imshow(pe, cmap="viridis")
plt.colorbar()
plt.title("Positional Encoding (50 posições × 16 dimensões)")
plt.show()

🟣 **Interpretação**

Cada linha = posição  
Cada coluna = dimensão do embedding  

O padrão:
- é suave  
- é periódico  
- permite ao modelo inferir distâncias  
- funciona para sequências maiores do que as vistas no treino  

---
# 4.4 Implementando Positional Encoding como camada Keras

In [ ]:
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_encoding = positional_encoding(max_len, d_model)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        return inputs + self.pos_encoding[:seq_len]

---
# 4.5 Testando a camada

In [ ]:
dummy = tf.random.normal((1, 10, 16))  # batch=1, tokens=10, embedding=16
pe_layer = PositionalEncoding(50, 16)
out = pe_layer(dummy)
out.shape

🟣 **Interpretação**

A saída tem o mesmo formato da entrada:

```
(batch, tokens, embedding)
```

Mas agora cada token contém:
- significado (embedding)
- posição (positional encoding)

---
# 4.6 Por que isso é tão poderoso?

✔️ Permite ordem sem recorrência  
✔️ Permite paralelização total  
✔️ Permite generalização para sequências maiores  
✔️ Permite que o modelo aprenda relações de distância  

Sem positional encoding, Transformers não funcionariam.

---
# 4.7 Conclusão desta parte

✔️ Entendemos por que Transformers precisam de posição  
✔️ Vimos o uso de seno/cosseno  
✔️ Visualizamos positional encoding  
✔️ Implementamos a camada em Keras  
✔️ Agora temos todos os blocos para montar um Transformer Encoder  

Agora estamos prontos para:

**PARTE 5 — Construindo um Transformer Encoder do zero.**

<a id="encoder"></a>
# 5. Construindo um Transformer Encoder do zero

Agora que entendemos:
- Self‑Attention  
- Multi‑Head Attention  
- Positional Encoding  

Vamos montar um **Encoder completo**, como no paper original.

Um bloco de Transformer Encoder contém:

1. Multi‑Head Self‑Attention  
2. Residual Connection  
3. Layer Normalization  
4. Feed‑Forward Network (FFN)  
5. Residual Connection  
6. Layer Normalization  

Vamos implementar tudo isso.

---
# 5.1 Multi‑Head Attention com Keras

In [ ]:
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.projection_dim = embed_dim // num_heads

        self.query = layers.Dense(embed_dim)
        self.key = layers.Dense(embed_dim)
        self.value = layers.Dense(embed_dim)

        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, Q, K, V):
        scores = tf.matmul(Q, K, transpose_b=True)
        scores = scores / tf.math.sqrt(tf.cast(self.projection_dim, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        Q = self.query(inputs)
        K = self.key(inputs)
        V = self.value(inputs)

        Q = self.separate_heads(Q, batch_size)
        K = self.separate_heads(K, batch_size)
        V = self.separate_heads(V, batch_size)

        attention = self.attention(Q, K, V)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat = tf.reshape(attention, (batch_size, -1, self.embed_dim))

        return self.combine_heads(concat)

---
# 5.2 Feed‑Forward Network (FFN)

In [ ]:
def feed_forward_network(embed_dim, ff_dim):
    return keras.Sequential([
        layers.Dense(ff_dim, activation="relu"),
        layers.Dense(embed_dim)
    ])

---
# 5.3 Construindo o bloco Transformer Encoder

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = feed_forward_network(embed_dim, ff_dim)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        # 1. Self‑Attention + Residual
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        # 2. Feed‑Forward + Residual
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

---
# 5.4 Testando o Transformer Encoder

In [ ]:
dummy = tf.random.normal((1, 10, 32))  # batch=1, tokens=10, embedding=32
encoder = TransformerEncoder(embed_dim=32, num_heads=4, ff_dim=64)
out = encoder(dummy)
out.shape

🟣 **Interpretação**

A saída tem o mesmo formato da entrada:

```
(batch, tokens, embedding)
```

Mas agora cada token:
- contém informações de todos os outros tokens  
- passou por atenção multi‑head  
- passou por feed‑forward  
- foi normalizado  
- tem residual connections  

Isso é exatamente o que acontece em BERT e GPT.

---
# 5.5 Visualização do fluxo interno

import matplotlib.pyplot as plt

plt.figure(figsize=(10,3))
plt.text(0.1, 0.5, "[Self‑Attention]", fontsize=14)
plt.text(3.5, 0.5, "[Residual + Norm]", fontsize=14)
plt.text(6.5, 0.5, "[Feed‑Forward]", fontsize=14)
plt.text(9.5, 0.5, "[Residual + Norm]", fontsize=14)
plt.axis("off")
plt.title("Fluxo interno de um Transformer Encoder")
plt.show()

---
# 5.6 Por que isso é tão poderoso?

✔️ Captura dependências longas  
✔️ Paralelização total  
✔️ Escala para bilhões de parâmetros  
✔️ Base de todos os modelos modernos  
✔️ Simples, modular e eficiente  

Um único bloco Encoder já é extremamente poderoso.

---
# 5.7 Conclusão desta parte

✔️ Implementamos Multi‑Head Attention  
✔️ Implementamos Feed‑Forward Network  
✔️ Implementamos Residual + LayerNorm  
✔️ Construímos um Transformer Encoder completo  
✔️ Testamos e entendemos o fluxo interno  

Agora estamos prontos para:

**PARTE 6 — Treinando um Transformer Encoder para classificação de texto.**

<a id="treino"></a>
# 6. Treinando um Transformer Encoder para Classificação de Texto

Agora vamos usar o Transformer Encoder que construímos para resolver um problema real:

**Classificação de sentimentos no dataset IMDB (positivo/negativo)**.

Este é o mesmo dataset usado com LSTM no módulo anterior — agora você verá como Transformers lidam com ele.

---
# 6.1 Carregando o dataset IMDB

In [ ]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

num_words = 10000
max_len = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

X_train.shape, X_test.shape

---
# 6.2 Criando Embedding + Positional Encoding

In [ ]:
embed_dim = 32
num_heads = 4
ff_dim = 64

inputs = keras.Input(shape=(max_len,))
x = layers.Embedding(num_words, embed_dim)(inputs)

# Positional Encoding
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_encoding = self.positional_encoding(max_len, d_model)

    def positional_encoding(self, max_len, d_model):
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(d_model)[np.newaxis, :]
        angles = pos / np.power(10000, (2 * (i//2)) / d_model)
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])
        return tf.cast(angles, dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:tf.shape(inputs)[1]]

x = PositionalEncoding(max_len, embed_dim)(x)

---
# 6.3 Adicionando o Transformer Encoder

In [ ]:
# Reutilizando a classe criada na parte 5
class MultiHeadSelfAttention(layers.Layer):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.projection_dim = embed_dim // num_heads

        self.query = layers.Dense(embed_dim)
        self.key = layers.Dense(embed_dim)
        self.value = layers.Dense(embed_dim)

        self.combine_heads = layers.Dense(embed_dim)

    def attention(self, Q, K, V):
        scores = tf.matmul(Q, K, transpose_b=True)
        scores = scores / tf.math.sqrt(tf.cast(self.projection_dim, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, V)

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        Q = self.query(inputs)
        K = self.key(inputs)
        V = self.value(inputs)

        Q = self.separate_heads(Q, batch_size)
        K = self.separate_heads(K, batch_size)
        V = self.separate_heads(V, batch_size)

        attention = self.attention(Q, K, V)
        attention = tf.transpose(attention, perm=[0, 2, 1, 3])
        concat = tf.reshape(attention, (batch_size, -1, self.embed_dim))

        return self.combine_heads(concat)

def feed_forward_network(embed_dim, ff_dim):
    return keras.Sequential([
        layers.Dense(ff_dim, activation="relu"),
        layers.Dense(embed_dim)
    ])

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = feed_forward_network(embed_dim, ff_dim)

        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)

        return out2

# Aplicando o encoder
x = TransformerEncoder(embed_dim, num_heads, ff_dim)(x)

---
# 6.4 Classificador final

In [ ]:
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

---
# 6.5 Treinando o Transformer

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

---
# 6.6 Avaliação no conjunto de teste

In [ ]:
loss, acc = model.evaluate(X_test, y_test, verbose=0)
acc

🟣 **Resultado típico**

- Acurácia entre **86% e 89%**  
- Similar ou superior à LSTM  
- Treinamento mais rápido  
- Melhor captura de contexto  

Mesmo um Transformer pequeno já supera muitos modelos clássicos.

---
# 6.7 Fazendo previsões em novos textos

In [ ]:
word_index = imdb.get_word_index()

def encode_text(text):
    tokens = []
    for word in text.lower().split():
        if word in word_index and word_index[word] < num_words:
            tokens.append(word_index[word])
        else:
            tokens.append(2)  # <UNK>
    return pad_sequences([tokens], maxlen=max_len)

sample = "This movie was absolutely fantastic and I loved it"
sample_encoded = encode_text(sample)

pred = model.predict(sample_encoded)[0][0]
pred

🟣 **Interpretação**

Valores próximos de:
- 1 → sentimento positivo  
- 0 → sentimento negativo  

---
# 6.8 Conclusão desta parte

✔️ Construímos um Transformer Encoder completo  
✔️ Aplicamos positional encoding  
✔️ Treinamos um modelo real de classificação  
✔️ Avaliamos e fizemos previsões  
✔️ Vimos que Transformers superam LSTMs em NLP  

Agora estamos prontos para:

**PARTE 7 — Transformers Pré‑Treinados (BERT, GPT, T5).**

<a id="pretreinados"></a>
# 7. Transformers Pré‑Treinados (BERT, GPT, T5)

Agora que você já construiu um Transformer Encoder do zero,
vamos entender os modelos que dominaram o NLP moderno.

Eles são chamados de **pré‑treinados** porque:
- foram treinados em bilhões de palavras
- aprenderam linguagem de forma geral
- podem ser ajustados (fine‑tuning) para tarefas específicas

Os três mais importantes:
- **BERT** → Encoder puro  
- **GPT** → Decoder puro  
- **T5** → Encoder + Decoder (seq2seq)  

Vamos entender cada um.

---
# 7.1 BERT — Bidirectional Encoder Representations from Transformers

BERT usa **apenas o Encoder** do Transformer.

Características:
- bidirecional (olha para esquerda e direita)
- excelente para compreensão de texto
- ótimo para:
  - classificação
  - análise de sentimentos
  - NER (entidades)
  - perguntas e respostas

Treinamento:
- Masked Language Modeling (MLM)
- Next Sentence Prediction (NSP)

---
# 7.2 GPT — Generative Pre‑trained Transformer

GPT usa **apenas o Decoder** do Transformer.

Características:
- unidirecional (olha só para o passado)
- especializado em geração de texto
- ótimo para:
  - completar frases
  - escrever textos
  - gerar código
  - chatbots

Treinamento:
- Language Modeling (prever o próximo token)

---
# 7.3 T5 — Text‑to‑Text Transfer Transformer

T5 usa **Encoder + Decoder**.

Ideia:
> “Tudo é texto para texto.”

Exemplos:
- tradução  
- sumarização  
- classificação  
- perguntas e respostas  
- correção gramatical  

Tudo é tratado como:

```
entrada: "traduza: Olá mundo"
saída:   "Hello world"
```

---
# 7.4 Usando modelos pré‑treinados com HuggingFace

Vamos usar:
- BERT para classificação
- GPT‑2 para geração
- T5 para sumarização

(Modelos pequenos para rodar localmente.)

In [ ]:
!pip install transformers --quiet

from transformers import pipeline

---
# 7.5 BERT — Classificação de Sentimentos

In [ ]:
classifier = pipeline("sentiment-analysis")
classifier("I really loved this movie, it was fantastic!")

---
# 7.6 GPT‑2 — Geração de Texto

In [ ]:
generator = pipeline("text-generation", model="gpt2")
generator("Once upon a time in a distant galaxy", max_length=40)

---
# 7.7 T5 — Sumarização

In [ ]:
summarizer = pipeline("summarization", model="t5-small")

text = """
Transformers revolutionized natural language processing by introducing the attention mechanism,
which allows models to capture long-range dependencies without recurrence.
This architecture enabled the development of large-scale language models.
"""

summarizer(text)

---
# 7.8 Comparação entre BERT, GPT e T5

| Modelo | Arquitetura | Direção | Melhor em |
|--------|-------------|----------|-----------|
| **BERT** | Encoder | bidirecional | compreensão |
| **GPT** | Decoder | unidirecional | geração |
| **T5** | Encoder‑Decoder | bidirecional + autoregressivo | tarefas gerais |

Cada um tem seu papel no ecossistema moderno.

---
# 7.9 Por que esses modelos mudaram tudo?

✔️ Aprendem linguagem geral  
✔️ Transferem conhecimento para qualquer tarefa  
✔️ Escalam para bilhões de parâmetros  
✔️ São base de modelos como:
  - BERT → RoBERTa, DistilBERT, ALBERT  
  - GPT → GPT‑2, GPT‑3, GPT‑4  
  - T5 → FLAN‑T5  

Eles são o alicerce da IA moderna.

---
# 7.10 Conclusão desta parte

✔️ Entendemos BERT, GPT e T5  
✔️ Vimos como cada um funciona  
✔️ Usamos modelos pré‑treinados com HuggingFace  
✔️ Entendemos o papel de cada arquitetura  

Agora estamos prontos para:

**PARTE 8 — Projeto Final: Mini‑GPT para completar frases.**

<a id="projeto"></a>
# 8. Projeto Final — Mini‑GPT para completar frases

Agora vamos construir um **modelo gerador de texto** baseado em:

- Transformer Decoder (simplificado)
- Self‑Attention mascarado
- Treinamento autoregressivo

Objetivo:
> Dado um início de frase, o modelo deve prever o próximo token.

Exemplo:
Entrada:
```
"o gato subiu"
```
Saída esperada:
```
"no"
```

---
# 8.1 Dataset simples de texto

Vamos usar um corpus pequeno para fins didáticos.

In [ ]:
text = """
o gato subiu no telhado
o gato pulou do muro
o cachorro latiu para o gato
o gato correu para o jardim
o cachorro correu atrás do gato
o gato dormiu no sofá
"""

---
# 8.2 Tokenização

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

sequences = []

words = text.split()
for i in range(3, len(words)):
    seq = words[i-3:i+1]
    sequences.append(tokenizer.texts_to_sequences([" ".join(seq)])[0])

sequences[:5]

---
# 8.3 Preparando dados para treino autoregressivo

In [ ]:
sequences = np.array(sequences)
X, y = sequences[:, :-1], sequences[:, -1]

X = pad_sequences(X, maxlen=3)

X.shape, y.shape

---
# 8.4 Máscara causal (evita olhar para o futuro)

GPT usa **máscara triangular** para impedir que o modelo veja tokens futuros.

In [ ]:
def causal_mask(seq_len):
    mask = np.triu(np.ones((seq_len, seq_len)), k=1)
    return tf.cast(mask == 0, tf.int32)

causal_mask(5)

---
# 8.5 Implementando o Decoder simplificado

In [ ]:
class MiniGPTDecoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x):
        seq_len = tf.shape(x)[1]
        mask = causal_mask(seq_len)

        attn = self.att(x, x, attention_mask=mask)
        x = self.norm1(x + attn)

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)

        return x

---
# 8.6 Construindo o Mini‑GPT completo

In [ ]:
embed_dim = 32
num_heads = 4
ff_dim = 64
max_len = 3

inputs = keras.Input(shape=(max_len,))
x = layers.Embedding(vocab_size, embed_dim)(inputs)

# Positional Encoding simples (aprendido)
x = layers.Embedding(max_len, embed_dim)(tf.range(max_len)) + x

decoder = MiniGPTDecoder(embed_dim, num_heads, ff_dim)
x = decoder(x)

x = layers.Flatten()(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.summary()

---
# 8.7 Treinando o Mini‑GPT

In [ ]:
history = model.fit(
    X, y,
    epochs=200,
    verbose=0
)

---
# 8.8 Função para prever o próximo token

In [ ]:
def predict_next(words):
    seq = tokenizer.texts_to_sequences([words])[0]
    seq = pad_sequences([seq], maxlen=3)
    pred = model.predict(seq, verbose=0)
    idx = np.argmax(pred)
    for w, i in word_index.items():
        if i == idx:
            return w

predict_next("o gato subiu")

---
# 8.9 Gerando texto automaticamente

In [ ]:
def generate_text(seed, steps=10):
    generated = seed
    for _ in range(steps):
        next_word = predict_next(generated.split()[-3:])
        generated += " " + next_word
    return generated

generate_text("o gato subiu", steps=8)

---
# 8.10 Conclusões do Projeto Final

✔️ Construímos um Transformer Decoder simplificado  
✔️ Implementamos máscara causal (como GPT)  
✔️ Treinamos um modelo autoregressivo  
✔️ Previsão de próximo token  
✔️ Geração automática de texto  
✔️ Criamos um Mini‑GPT funcional  

Você agora entende:
- como GPT funciona internamente  
- como Transformers geram texto  
- como treinar modelos autoregressivos  

E com isso, você conclui o **Módulo 14 — Transformers**.

O próximo passo é o **Módulo 15 — Fine‑Tuning, LoRA e Instrução de Modelos**.